In [1]:
import numpy as np
import pandas as pd

In [5]:
df=pd.read_csv(r"D:\PROJECT DATA ANALYSIS\messed_Product.csv")

In [6]:
df.info()
df.isnull().sum()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ProductKey               387 non-null    float64
 1   Product                  381 non-null    str    
 2   Standard Cost            387 non-null    str    
 3   Color                    334 non-null    str    
 4   Subcategory              378 non-null    str    
 5   Category                 384 non-null    str    
 6   Background Color Format  389 non-null    str    
 7   Font Color Format        391 non-null    str    
dtypes: float64(1), str(7)
memory usage: 25.4 KB


,ProductKey,Product,Standard Cost,Color,Subcategory,Category,Background Color Format,Font Color Format
0,210.0,"HL Road Frame - Black, 58",$868.63,Black,Road Frames,Components,#000000,#FFFFFF
1,215.0,"sport-100 helmet, black",$12.03,Black,Helmets,Accessories,#000000,#FFFFFF
2,216.0,"Sport-100 Helmet, Black",$13.88,Black,NaN,Accessories,#000000,#FFFFFF
3,217.0,"Sport-100 Helmet, Black",$13.09,Black,Helmets,Accessories,#000000,#FFFFFF
4,253.0,"ll road frame - black, 58",NaN,Black,NaN,NaN,#000000,#FFFFFF


In [7]:
import re

text_cols = df.select_dtypes(include="object").columns
for c in text_cols:
    df[c] = df[c].str.strip()
    df[c] = df[c].replace("", np.nan)

C:\Users\Ahmed\AppData\Local\Temp\ipykernel_16992\1872521430.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns


In [8]:
def parse_currency(value):
    if pd.isna(value):
        return np.nan
    cleaned = re.sub(r"[^0-9.\-]", "", str(value))
    if cleaned in ("", "-", "."):
        return np.nan
    try:
        return float(cleaned)
    except ValueError:
        return np.nan

df["Standard Cost"] = df["Standard Cost"].apply(parse_currency)

In [9]:
for col in ["Color", "Subcategory", "Category"]:
    df[col] = df[col].str.strip().str.title()

In [10]:
def smart_title(text, uppercase_tokens=frozenset({"hl", "ll", "ml", "xl"})):
    if pd.isna(text):
        return text
    text = text.strip()
    if text != text.lower():
        return text
    fixed = []
    for w in text.split(" "):
        letters = re.sub(r"[^A-Za-z]", "", w).lower()
        fixed.append(w.upper() if letters in uppercase_tokens else w.capitalize())
    return " ".join(fixed)

df["Product"] = df["Product"].apply(smart_title)

In [11]:
for col in ["Background Color Format", "Font Color Format"]:
    df[col] = df[col].str.strip().str.upper()

In [12]:
df["ProductKey"] = pd.to_numeric(df["ProductKey"], errors="coerce")
df["ProductKey"] = np.abs(df["ProductKey"])
df = df[df["ProductKey"].notna()].copy()
df["ProductKey"] = df["ProductKey"].astype("Int64")

In [13]:
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["ProductKey"], keep="first")
df = df.sort_values("ProductKey").reset_index(drop=True)

In [14]:
df.info()
df.isnull().sum()
df.head(10)

<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ProductKey               380 non-null    Int64  
 1   Product                  359 non-null    str    
 2   Standard Cost            365 non-null    float64
 3   Color                    313 non-null    str    
 4   Subcategory              354 non-null    str    
 5   Category                 361 non-null    str    
 6   Background Color Format  366 non-null    str    
 7   Font Color Format        368 non-null    str    
dtypes: Int64(1), float64(1), str(6)
memory usage: 24.2 KB


,ProductKey,Product,Standard Cost,Color,Subcategory,Category,Background Color Format,Font Color Format
0,210,"HL Road Frame - Black, 58",868.63,Black,Road Frames,Components,#000000,#FFFFFF
1,211,"HL Road Frame - Red, 58",868.63,Red,Road Frames,Components,#FF0000,#FFFFFF
2,212,"Sport-100 Helmet, Red",12.03,Red,Helmets,Accessories,#FF0000,#FFFFFF
3,213,"Sport-100 Helmet, Red",13.88,Red,Helmets,Accessories,#FF0000,#FFFFFF
4,214,"Sport-100 Helmet, Red",13.09,Red,Helmets,Accessories,#FF0000,#FFFFFF
5,215,"Sport-100 Helmet, Black",12.03,Black,Helmets,Accessories,#000000,#FFFFFF
6,216,"Sport-100 Helmet, Black",13.88,Black,NaN,Accessories,#000000,#FFFFFF
7,217,"Sport-100 Helmet, Black",13.09,Black,Helmets,Accessories,#000000,#FFFFFF
8,218,"Mountain Bike Socks, M",3.40,White,Socks,Clothing,#FFFFFF,#000000
9,219,"Mountain Bike Socks, L",3.40,White,Socks,Clothing,#FFFFFF,#000000


In [15]:
df.to_csv(r"D:\PROJECT DATA ANALYSIS\Product_cleaned.csv", index=False)